In [5]:
import pandas as pandas
import numpy as np

In [10]:
df_raw = pd.Dataframe([
    {"pedido_id": "P001", "tipo_entrega": "Moto", "tempo_entrega": 24, "valor_pedido": 45.0, "avaliacao": 5.0},
    {"pedido_id": "P002", "tipo_entrega": "Carro", "tempo_entrega": 26, "valor_pedido": np.nan, "avaliacao": 4.0},
    {"pedido_id": "P003", "tipo_entrega": "Moto", "tempo_entrega": 28, "valor_pedido": 62.0, "avaliacao": 5.0},
    {"pedido_id": "P004", "tipo_entrega": "Bicicleta", "tempo_entrega": 30, "valor_pedido": 38.0, "avaliacao": 3.0},
    {"pedido_id": "P005", "tipo_entrega": "Carro", "tempo_entrega": 32, "valor_pedido": 80.0, "avaliacao": np.nan},
    {"pedido_id": "P006", "tipo_entrega": "Moto", "tempo_entrega": 180, "valor_pedido": 55.0, "avaliacao": 2.0},
    {"pedido_id": "P006", "tipo_entrega": "Bicicleta", "tempo_entrega": 30, "valor_pedido": 38.0, "avaliacao": 3.0}
])

df_raw

NameError: name 'pd' is not defined

In [ ]:
print("Total de linhas:", len(df_raw))
print("Duplicadas:", df_raw.duplicated().sum())
print("Nulos por coluna:")
print(df_raw.isna().sum())

In [ ]:
# Tratamento desses vaores Nulos
# Remove duplicadas e preenche nulos numéricos com mediana
df_clean = df_raw.drop_duplicates().copy()

for col in ["tempo_entrega", "valor_pedido", "avaliacao"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean

In [ ]:
# Tratar outlier em tempo de entrega usando quantis(Q1 e Q3)

# Quantil é um ponto de corte da distribuição
# - quantile(0.25) => Q1: valor abaixo de ~25%
# - quantile(0.75) => Q3: valor abaixo de ~75%
q1 = df_clean["tempo_entrega"].quantile(0.25)
q3 = df_clean["tempo_entrega"].quantile(0.75)

# IQR (Intervalo interquartil) mede a faixa central dos dados
# IQR = Q3 - Q1
iqr = q3 - q1

# Limites clássicos
# Valor abaixo de lower ou acima de upper considerado discrepante
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_clean["tempo_entrega_tratado"] = df_clean["tempo_entrega"].clip(lower=lower, upper=upper)

alterados = (df_clean["tempo_entrega"] != df_clean["tempo_entrega_tratado"].sum())

print(f"Q1 (25%): {q1:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Limite inferior: {lower:.2f}")
print(f"Limite superior: {upper:.2f}")
print(f"Tempos alterados pelo clipping: {alterados}")

df_clean[["pedido_id", "tempo_entrega", "tempo_entrega_tratado"]]


: 

In [ ]:
# Relacao entre colunas numericas

# Correlacao indica como duas variaveis mudam juntas.
# Valor proximo de +1: quando uma sobe, a outra tende a subir.
# Valor proximo de -1: quando uma sobe, a outra tende a descer.
# Valor proximo de 0: pouca relacao linear entre elas.
df_clean[["tempo_entrega_tratado", "valor_pedido", "avaliacao"]].corr()